# C-6b: escenarios de liquidez escalando solo al market maker (BORRADOR de PS para BF)

El C-6 anterior dio **REVISAR**: escalar a los agentes de ruido y de valor mezcló liquidez con volatilidad y rompió el pareo por semilla (`docs/c6_liquidez_resultado_PS.md`). Aquí se escala **solo `mm_pov`** (profundidad del market maker): 0,25×, 1× y 4× (`data/scenarios/escenarios_liquidez_v2.json`), con **60 semillas**.

El criterio quedó fijado antes de correr: se juzga con comprar 5.000 acciones a mercado, que es lo que más depende de la profundidad del libro.

**Cómo usarlo:** «Ejecutar todas». Dura ≈ **1 h 20 min**. Si se corta, vuelve a ejecutar todo: retoma donde quedó. Al final imprime **OK** o **REVISAR**.

Todo queda en **`slippage_colab/resultados/c6b_liquidez`** y se arma `c6b_liquidez.zip`.

## Preparación del entorno (celdas 1 a 9)

In [ ]:
# CELDA 1 (reinicia el entorno solo; el aviso "sesión fallada" es normal, no la repitas)
!pip install -q "condacolab<0.2"
import condacolab
condacolab.install()

In [ ]:
# CELDA 2
import os
if not os.path.exists('/usr/local/envs/abides'):
    !conda create -n abides python=3.9 -y -q
!conda run -n abides python --version

In [ ]:
# CELDA 3 (unos minutos; un aviso de Pillow es normal)
%env PIP_DISABLE_PIP_VERSION_CHECK=1
!conda run -n abides python -m pip install -q pip==23.3.2 setuptools==57.5.0 wheel==0.38.4
!conda run -n abides python -m pip install -q "cython<3" numpy==1.22.0
!conda run -n abides python -m pip install -q --no-build-isolation --no-deps gym==0.18.0
!conda run -n abides python -m pip install -q --no-build-isolation numpy==1.22.0 "cloudpickle>=1.2,<1.7" "pyglet>=1.4,<=1.5.0" "Pillow<10" pandas==1.2.4 scipy==1.10.0 coloredlogs==15.0.1 psutil==5.8.0 tqdm==4.61.1 pomegranate==0.14.5 "ray[tune]==1.7.0" "protobuf==3.20.3" pyarrow==11.0.0 matplotlib==3.5.3 pytest
!conda run -n abides python -m pip install -q "gymnasium==0.29.1" "importlib-metadata>=4.8" farama-notifications --no-deps

In [ ]:
# CELDA 4 (deben salir tres "OK")
import os
if not os.path.exists('/content/abides-jpmc-public'):
    !git clone -q https://github.com/jpmorganchase/abides-jpmc-public.git /content/abides-jpmc-public
for pkg in ['abides-core', 'abides-markets', 'abides-gym']:
    !cd /content/abides-jpmc-public/{pkg} && conda run -n abides python setup.py -q install > /content/install_{pkg}.log 2>&1 && echo "{pkg} OK" || (echo "{pkg} FALLO"; tail -20 /content/install_{pkg}.log)

In [ ]:
# CELDA 5 (debe imprimir 0.18.0 1.22.0 1.2.4 0.29.1 1.7.0)
!conda run -n abides python -c "import gym, numpy, pandas, gymnasium, ray, abides_core, abides_markets, abides_gym; print(gym.__version__, numpy.__version__, pandas.__version__, gymnasium.__version__, ray.__version__)"

In [ ]:
# CELDA 6 (torch; debe imprimir torch 1.13.1+cpu | numpy 1.22.0)
!conda run -n abides python -m pip install -q torch==1.13.1+cpu --extra-index-url https://download.pytorch.org/whl/cpu
!conda run -n abides python -c "import torch, numpy; print('torch', torch.__version__, '| numpy', numpy.__version__)"

In [ ]:
# CELDA 7 (código del proyecto: rama feature/ps-borrador-bf-c6b-mm; debe mostrar un commit y un número mayor que 0)
import os
if not os.path.exists('/content/slippage'):
    !git clone -q https://github.com/Vatelfil/slippage.git /content/slippage
!cd /content/slippage && git fetch -q origin && git checkout -q -f origin/feature/ps-borrador-bf-c6b-mm
!cd /content/slippage && git log --oneline -1 && grep -c "escenarios_liquidez" src/envs/escenarios_liquidez.py


In [ ]:
# CELDA 8 (conecta Drive; acepta el permiso)
from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = '/content/drive/MyDrive/slippage_colab/resultados'
import os; os.makedirs(RESULTS_DIR, exist_ok=True)
print(RESULTS_DIR)

In [ ]:
# CELDA 9 (seaborn, que necesita el módulo de benchmarks)
!conda run -n abides python -m pip install -q seaborn==0.12.2 --no-deps

## Corrida

In [ ]:
# 3 escenarios × 2 heurísticas × 2 tamaños × 60 semillas (≈ 1 h 20 min)
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/c6_escenarios_liquidez_bf.py --escenarios-json data/scenarios/escenarios_liquidez_v2.json --n-semillas 60 --out {RESULTS_DIR}/c6b_liquidez/c6b_liquidez.json 2>&1 | grep -v UserWarning


## Zip para descargar

In [ ]:
# CELDA FINAL: zip pequeño (sin checkpoints) para pasárselo a PS
import os, zipfile
base = f'{RESULTS_DIR}/c6b_liquidez'
archivos = [os.path.join(c, a) for c, _, fs in os.walk(base) if 'checkpoints' not in c for a in fs if a.endswith(('.json', '.csv', '.md'))]
print(len(archivos), 'archivos')
zpath = f'{RESULTS_DIR}/c6b_liquidez.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for a in archivos:
        z.write(a, os.path.relpath(a, RESULTS_DIR))
print('zip listo:', zpath, '-> descárgalo desde slippage_colab/resultados en Drive')
